# (21) conv latents:  phi working?

**Motivation**: host = ```any```, device = ```cuda:0``` <br>

In [1]:
# HIDE CODE


import os, sys
from IPython.display import display

# tmp & extras dir
git_dir = os.path.join(os.environ['HOME'], 'Dropbox/git')
extras_dir = os.path.join(git_dir, 'jb-progress-2025/_extras')
fig_base_dir = os.path.join(git_dir, 'jb-progress-2025/figs')
tmp_dir = os.path.join(git_dir, 'jb-progress-2025/tmp')

# GitHub
sys.path.insert(0, os.path.join(git_dir, '_TemporalSC'))
from figures.convergence import plot_convergence
from main.config_defaults import default_configs
from figures.fighelper import *
from main.train import *

# warnings, tqdm, & style
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
from rich.jupyter import print
%matplotlib inline
set_style()

In [2]:
device_idx = 0
device = f'cuda:{device_idx}'

print(f"device: {device}  ———  host: {os.uname().nodename}")

device: cuda:0  ———  host: mach

## Make a model

In [3]:
from base.helper import conv_arithmetic
from main.layers import ConvDictionary

## conv arithmetic

In [4]:
df = conv_arithmetic(
    input_size=8,
    output_size=16,
    # kernel_size=None,
    stride=1,
    padding=0,
    deconv=True,
)
df

,mode,computed,parameter,value
0,deconv,False,input_size,8
1,deconv,False,output_size,16
2,deconv,True,kernel_size,9
3,deconv,False,stride,1
4,deconv,False,padding,0


In [5]:
print(df.to_string(index=False))

mode  computed   parameter  value
deconv     False  input_size      8
deconv     False output_size     16
deconv      True kernel_size      9
deconv     False      stride      1
deconv     False     padding      0

In [6]:
df.loc[df['computed']]

,mode,computed,parameter,value
2,deconv,True,kernel_size,9


In [7]:
kws = dict(
    in_channels=512,
    out_channels=1,
    kernel_size=9,
)

phi = ConvDictionary(**kws)
deconv_torch = nn.ConvTranspose2d(**kws)

phi.weight.shape, deconv_torch.weight.shape

(torch.Size([512, 1, 9, 9]), torch.Size([512, 1, 9, 9]))

In [8]:
z = torch.randn((123, 512, 8, 8))
x_hat = phi(z)
x_hat.shape

torch.Size([123, 1, 16, 16])

In [9]:
x = torch.randn((123, 1, 16, 16))
delta = x - x_hat
delta.shape

torch.Size([123, 1, 16, 16])

In [10]:
du = phi(delta, encoding=True)
du.shape

torch.Size([123, 512, 8, 8])

In [11]:
print(phi)

ConvDictionary(in_channels=512, out_channels=1, kernel_size=9)

In [12]:
df = conv_arithmetic(
    input_size=4,
    output_size=16,
    # kernel_size=None,
    stride=2,
    padding=0,
    deconv=True,
)
df

,mode,computed,parameter,value
0,deconv,False,input_size,4
1,deconv,False,output_size,16
2,deconv,True,kernel_size,10
3,deconv,False,stride,2
4,deconv,False,padding,0


In [13]:
df.loc[df['computed']]

,mode,computed,parameter,value
2,deconv,True,kernel_size,10


In [14]:
kws = dict(
    in_channels=512,
    out_channels=1,
    kernel_size=10,
    stride=2,
    padding=0,
)

phi = ConvDictionary(**kws)
z = torch.randn((123, 512, 4, 4))
x_hat = phi(z)
x_hat.shape

torch.Size([123, 1, 16, 16])

In [15]:
du = phi(x_hat, encoding=True)
du.shape

torch.Size([123, 512, 4, 4])

In [16]:
print(phi)

ConvDictionary(in_channels=512, out_channels=1, kernel_size=10, stride=2)